In [ ]:
import glob

files_to_process = glob.glob('/mnt/f/multichain_dataset_20250917/PDB_ids/rcsb_pdb_ids_*.txt')
output_filename = "/mnt/f/multichain_dataset_20250917/PDB_ids/all_ids_one_per_line.txt"

all_pdb_ids = []

if not files_to_process:
    print("Error: The 'rcsb_pdb_ids_*.txt' file was not found in the current directory.")
    print("Make sure that this script and the ID file you downloaded are in the same folder.")
else:
    print(f"Found {len(files_to_process)} ID files that need to be processed...")

    for filename in files_to_process:
        print(f"Processing file: {filename}")
        with open(filename, 'r') as f:
            # Read the entire content of the file, which is only one line, and split it by commas
            content = f.read()
            ids_in_file = [pdb_id.strip() for pdb_id in content.split(',') if pdb_id.strip()]
            all_pdb_ids.extend(ids_in_file)

    print(f"\nProcessing completed! A total of {len(all_pdb_ids)} independent PDB IDs were found.")

    with open(output_filename, "w") as f:
        for pdb_id in all_pdb_ids:
            f.write(f"{pdb_id}\n")

    print(f"All IDs have been successfully written to the new file:'{output_filename}'")
    print("Now you can use this file to proceed with the next download.")

In [ ]:
import os
import subprocess
from tqdm import tqdm

ID_LIST_FILE = "/mnt/f/multichain_dataset_20250917/PDB_ids/all_ids_one_per_line.txt"
OUTPUT_DIR = "/mnt/f/multichain_dataset_20250917/mmcif_files"
FILE_FORMAT = "cif"
COMPRESSED = True
ARIA2C_URL_LIST_FILE = os.path.join("/mnt/f/multichain_dataset_20250917/aria2c_urls.txt")


def main():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    if not os.path.exists(ID_LIST_FILE):
        print(f"Error: Unable to locate the ID list file '{ID_LIST_FILE}'.")
        print("Make sure that the file name and path are correct.")
        return

    with open(ID_LIST_FILE, 'r') as f:
        required_ids = {line.strip().upper() for line in f if line.strip()}
    print(f"The program has read {len(required_ids)} unique PDB IDs from the '{ID_LIST_FILE}' file.")

    file_suffix = f".{FILE_FORMAT}"
    if COMPRESSED:
        file_suffix += ".gz"

    # Generate a list of URLs for use by aria2c
    print(f"Generating a list of download URLs for {len(required_ids)} PDB IDs...")
    with open(ARIA2C_URL_LIST_FILE, 'w') as f:
        for pdb_id in tqdm(required_ids, desc="Generating URLs"):
            pdb_id_lower = pdb_id.lower()
            download_url = f"https://files.rcsb.org/download/{pdb_id_lower}{file_suffix}"
            f.write(download_url + "\n")

    print(f"The URL list has been saved to '{ARIA2C_URL_LIST_FILE}'.")

    aria2c_command = [
        "aria2c",
        "-i", ARIA2C_URL_LIST_FILE,
        "-d", OUTPUT_DIR,
        "-c",
        "-x", "16",
        "-s", "16",
    ]

    print("\n" + "=" * 20 + " Start using aria2c for downloading " + "=" * 20)
    print("The download progress will be displayed by the aria2c console.")
    print("=" * 50)

    try:
        subprocess.run(aria2c_command, check=True)
    except FileNotFoundError:
        print("\nError: The 'aria2c' command cannot be found. Please ensure that it is installed and added to the system PATH.")
    except subprocess.CalledProcessError as e:
        print(f"\nError: The aria2c process returned a non-zero exit code {e.returncode}。")
        print("Please check the output of aria2c to obtain more information.")

    print("\n" + "=" * 20 + " Download task completed " + "=" * 20)

    # os.remove(ARIA2C_URL_LIST_FILE)
    # print(f"The URL list file has been deleted: {ARIA2C_URL_LIST_FILE}")


if __name__ == "__main__":
    main()

In [ ]:
'''To check whether all the files have been downloaded completely'''

import os
from pathlib import Path

id_list_file = Path(r"F:\multichain_dataset_20250917\PDB_ids\all_ids_one_per_line.txt")
download_dir = Path(r"F:\multichain_dataset_20250917\mmcif_files")

# --- Read the list of desired PDB IDs ---
try:
    with open(id_list_file, 'r') as f:
        # Read each line, remove leading and trailing whitespaces, convert to lowercase, and store in a set for quick lookup
        expected_ids = {line.strip().lower() for line in f if line.strip()}
    print(f"The list file contains {len(expected_ids)} expected PDB IDs.")
except FileNotFoundError:
    print(f"Error: The ID list file was not found. Please check the path: '{id_list_file}'")
    exit()  # If the file does not exist, exit the script.

# --- Scan the actual downloaded files ---
if not download_dir.is_dir():
    print(f"Error: The download directory was not found. Please check the path:'{download_dir}'")
    exit()

# Traverse the folder, extract the file name (for example, '117e.cif.gz'),
# Then split and take the first part ('117e'), and convert it to lowercase
actual_ids = {file.name.split('.')[0].lower() for file in download_dir.glob('*.cif.gz')}
print(f"In the download directory, {len(actual_ids)} .cif.gz files were found.")


missing_ids = expected_ids - actual_ids

print("-" * 30)
if missing_ids:
    print(f"Screening is complete. {len(missing_ids)} missing files were found:")
    for pdb_id in sorted(list(missing_ids)):
        print(f"  - {pdb_id}")
else:
    print("The screening is complete and there are no missing documents.")

In [ ]:
import os
import pickle
import glob
import numpy as np
import prody
from tqdm import tqdm


RESTYPE_3_TO_1 = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q",
    "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K",
    "MET": "M", "PHE": "F", "PRO": "P", "SER": "S", "THR": "T", "TRP": "W",
    "TYR": "Y", "VAL": "V",
}
STANDARD_AMINO_ACIDS_3 = set(RESTYPE_3_TO_1.keys())
ATOM_ORDER = {
    "N": 0, "CA": 1, "C": 2, "CB": 3, "O": 4, "CG": 5, "CG1": 6, "CG2": 7,
    "OG": 8, "OG1": 9, "SG": 10, "CD": 11, "CD1": 12, "CD2": 13, "ND1": 14,
    "ND2": 15, "OD1": 16, "OD2": 17, "SD": 18, "CE": 19, "CE1": 20, "CE2": 21,
    "CE3": 22, "NE": 23, "NE1": 24, "NE2": 25, "OE1": 26, "OE2": 27, "CH2": 28,
    "NH1": 29, "NH2": 30, "OH": 31, "CZ": 32, "CZ2": 33, "CZ3": 34, "NZ": 35,
    "OXT": 36,
}
MAX_ATOMS = len(ATOM_ORDER)


def extract_chain_features(chain_selection: prody.Selection, chain_idx: int):
    
    # By this method, non-standard amino acids can be excluded, but this extraction method allows for the occurrence of chain breaks.
    residues = [res for res in chain_selection.getHierView().iterResidues() if
                res.getResname() in STANDARD_AMINO_ACIDS_3]  
    
    num_residues = len(residues)

    chain_id = list(set(chain_selection.getChids()))[0]

    xyz_37 = np.zeros((num_residues, MAX_ATOMS, 3), dtype=np.float32)
    mask_37 = np.zeros((num_residues, MAX_ATOMS), dtype=bool)
    res_indices = np.zeros(num_residues, dtype=np.int32)
    sequence_list = []

    for i, res in enumerate(residues):
        res_indices[i] = res.getResnum()
        sequence_list.append(RESTYPE_3_TO_1.get(res.getResname(), 'X'))
        for atom in res:
            atom_name = atom.getName()
            if atom_name in ATOM_ORDER:
                atom_idx = ATOM_ORDER[atom_name]
                xyz_37[i, atom_idx] = atom.getCoords()
                mask_37[i, atom_idx] = True
    sequence = "".join(sequence_list)
    backbone_mask = (mask_37[:, ATOM_ORDER['N']] & mask_37[:, ATOM_ORDER['CA']] & mask_37[:, ATOM_ORDER['C']] & mask_37[:, ATOM_ORDER['O']])
    return {
        'xyz_37': xyz_37,
        'xyz_37_mask': mask_37,
        'seq': sequence,
        'R_idx': res_indices,
        'chain_id': chain_id,
        'chain_idx': chain_idx,
        'mask': backbone_mask
    }


def process_structure(structure_obj: prody.AtomGroup, assembly_id_str: str, pkl_dir: str, pdb_id: str) -> bool:

    if not hasattr(structure_obj, 'numAtoms'):
        return False
    output_pkl_path = os.path.join(pkl_dir, f"{assembly_id_str}.pkl")
    if os.path.exists(output_pkl_path):
        return True
    clean_chains_selections = []

    try:
        unique_chids = sorted(list(set(structure_obj.getChids())))
    except AttributeError:
        return False

    for chid in unique_chids:
        chain_selection = structure_obj.select(f'chain {chid}')
        if not hasattr(chain_selection, 'numAtoms') or chain_selection.numAtoms() == 0:
            continue
        protein_only_selection = chain_selection.select('protein')
        if not hasattr(protein_only_selection, 'numAtoms') or protein_only_selection.numAtoms() == 0:
            continue
        is_clean = True
        try:
            for res in protein_only_selection.getHierView().iterResidues():
                if res is None:
                    is_clean = False
                    break
                if (res.getAtom('CA') is None or res.getAtom('N') is None
                        or res.getAtom('C') is None or res.getAtom('O') is None):
                    is_clean = False
                    break
        except Exception:
            is_clean = False
        if is_clean:
            clean_chains_selections.append(protein_only_selection)

    if len(clean_chains_selections) < 2:
        return False

    processed_chains_list = []
    sorted_clean_chains = sorted(clean_chains_selections, key=lambda c: list(set(c.getChids()))[0])
    chain_id_map = {list(set(ch.getChids()))[0]: idx for idx, ch in enumerate(sorted_clean_chains)}
    for chain_obj in sorted_clean_chains:
        current_chain_id = list(set(chain_obj.getChids()))[0]
        chain_features = extract_chain_features(chain_obj, chain_id_map[current_chain_id])
        processed_chains_list.append(chain_features)

    final_data_to_save = {
        'pdb_id': pdb_id,
        'chain_features': processed_chains_list
    }

    chain_ids_str = ''.join(sorted(chain_id_map.keys()))
    new_assembly_id_str = f"{assembly_id_str}_{chain_ids_str}"
    output_pkl_path = os.path.join(pkl_dir, f"{new_assembly_id_str}.pkl")

    if os.path.exists(output_pkl_path):
        return True

    with open(output_pkl_path, 'wb') as f:
        pickle.dump(final_data_to_save, f)

    return True


def process_cif_file(cif_filepath, pkl_dir, exclusion_list):

    pdb_id = os.path.basename(cif_filepath).split('.')[0].lower()

    # Check whether the PDB ID is included in the exclusion list
    if pdb_id in exclusion_list:
        tqdm.write(f"Skip: PDB ID {pdb_id} is already in the exclusion list.")
        return

    is_processed = False

    # plan A
    try:
        assemblies = prody.parsePDB(cif_filepath, biomol='all')
        assemblies_list = []
        if assemblies is not None:
            if isinstance(assemblies, prody.AtomGroup):
                assemblies_list = [assemblies]
            elif isinstance(assemblies, list):
                assemblies_list = [asm for asm in assemblies if isinstance(asm, prody.AtomGroup)]

        if assemblies_list:
            for i, biounit in enumerate(assemblies_list, 1):
                assembly_id_str = f"{pdb_id}_bio_{i}"
                if process_structure(biounit, assembly_id_str, pkl_dir, pdb_id):
                    is_processed = True
            if is_processed:
                return
    except Exception:
        pass

    # plan B
    if not is_processed:
        try:
            asu = prody.parsePDB(cif_filepath)
            if asu is None:
                return

            num_assemblies = asu.getBiomolAssemblyNum()
            if num_assemblies > 0:
                for i in range(1, num_assemblies + 1):
                    try:
                        biounit = prody.parsePDB(cif_filepath, biomol=True, model=i)
                        assembly_id_str = f"{pdb_id}_bio_{i}"
                        if process_structure(biounit, assembly_id_str, pkl_dir, pdb_id):
                            is_processed = True
                    except Exception:
                        continue
                if is_processed:
                    return
        except Exception:
            pass

    # plan C
    if not is_processed:
        try:
            if 'asu' not in locals() or asu is None:
                asu = prody.parsePDB(cif_filepath)

            if asu is not None:
                assembly_id_str = f"{pdb_id}_asu"
                process_structure(asu, assembly_id_str, pkl_dir, pdb_id)
            else:
                tqdm.write(f"Error: The file {pdb_id} could not be parsed by ProDy in the end.")
        except Exception as e:
            tqdm.write(f"An error occurred while processing the ASU for the file {pdb_id}: {e}")


def main():

    INPUT_DIR = "/mnt/f/multichain_dataset_20250917/test"
    OUTPUT_PKL_DIR = "/mnt/f/multichain_dataset_20250917/multiChain_processed_pkls"
    EXCLUSION_FILE = "/mnt/f/multichain_dataset_20250917/finetune_exclusion_pdb_ids.txt"

    os.makedirs(OUTPUT_PKL_DIR, exist_ok=True)

    exclusion_list = set()
    try:
        with open(EXCLUSION_FILE, 'r') as f:
            for line in f:
                exclusion_list.add(line.strip().lower())
        print(f"{len(exclusion_list)} PDB IDs have been loaded into the exclusion list.")
    except FileNotFoundError:
        print(f"Warning: The exclusion file '{EXCLUSION_FILE}' cannot be found. All files will be processed.")

    cif_files = glob.glob(os.path.join(INPUT_DIR, '*.cif.gz'))

    if not cif_files:
        print(f"Error: No .cif.gz files were found in the directory '{INPUT_DIR}'.")
        return

    print(f"Found {len(cif_files)} mmcif files ready for processing...")

    for cif_file in tqdm(cif_files, desc="Processing mmcif files"):
        process_cif_file(cif_file, OUTPUT_PKL_DIR, exclusion_list)

    print("\nAll the files have been processed!")
    print(f"The processed pickle file is saved in: {OUTPUT_PKL_DIR}")


if __name__ == "__main__":
    prody.confProDy(verbosity='none')
    main()